In [40]:
def collect_all_paths(folder, n_threads=2, FILTER_KEYS=[]):
    paths = old_pd.list_paths_in_partition()
    filtered_paths = paths
    if FILTER_KEYS:
        filtered_paths = [p for p in paths if any(key in p for key in FILTER_KEYS)]
    all_paths = [p for p in filtered_paths if not p.startswith("/silver/")]
    return all_paths

def split(i):
    if "=" not in i:
        return i
    return i.split("=")[1]

def migrate_one_file(old_pd, new_pd, force, p):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if "/silver/" in p:
                return {
                    "ok": False,
                    "path": p,
                    "error": "Not a valid path, SILVER",
                    "attempts": attempt,
                }
            # Rearrange path
            try:
                if "/raw/" in p:
                    tparts = p.replace("/raw/", "")
                    parts = tparts.split("/")
                    category      = split(parts[0])
                    module        = split(parts[1])
                    instance_name = split(parts[2])
                    year          = split(parts[3])
                    month         = split(parts[4])
                    day           = split(parts[5])
                    file_name     = split(parts[6])
                else:
                    tparts = p.lstrip("/")
                    parts = tparts.split("/")
                    instance_name = split(parts[0])
                    category      = split(parts[1])
                    module        = split(parts[2])
                    year          = split(parts[3])
                    month         = split(parts[4])
                    day           = split(parts[5])
                    file_name     = split(parts[6])
            except Exception as e:
                return {
                    "ok": False,
                    "path": p,
                    "error": str(e),
                    "attempts": attempt,
                }
            new_path = (
                f"raw/"
                f"category={category}/"
                f"module={module}/"
                f"instance_name={instance_name}/"
                f"year={year}/"
                f"month={month}/"
                f"day={day}/"
                f"{file_name}"
            )
            
            # Check if already exists
            if not force:
                try:
                    exists = new_pd.get_path_details(new_path).get("exists", False)
                except Exception:
                    exists = False  # fall back to attempting upload
                if exists:
                    return {
                        "ok": True,
                        "dst": new_path,
                        "attempts": 0,
                        "skipped": True,
                    }

            # Read old parquet
            with old_pd.get_download_stream(p) as stream:
                file_bytes = io.BytesIO(stream.read())
            df = pd.read_parquet(file_bytes)
            
            # add in timestamp for missing df for operating system stuff
            if category == "operating_system" and (
                        "timestamp" not in df.columns or df["timestamp"].isna().all()
                    ):
                date = f"{year}/{month}/{day}"
                df["timestamp"] = pd.to_datetime(date, format="%Y/%m/%d", utc=True)

            # Write to new location
            buffer = io.BytesIO()
            df.to_parquet(
                buffer,
                compression="gzip",
                engine="pyarrow",
                index=False,
            )
            buffer.seek(0)
            content = buffer.read()
            new_pd.upload_stream(new_path, content)

            return {
                "ok": True,
                "dst": new_path,
                "attempts": attempt,
            }
        
        except Exception as e:
            if attempt == MAX_RETRIES:
                return {
                    "ok": False,
                    "path": p,
                    "error": str(e),
                    "attempts": attempt,
                }

            # Exponential backoff + jitter
            sleep_time = (2 ** attempt) + random.random()
            time.sleep(sleep_time)
    return

In [26]:
# --------------------------------------------------
# MAIN
# --------------------------------------------------
import io
import os
import time
import pandas as pd
import dataiku
from joblib import Parallel, delayed
import time
import random


old_pd = dataiku.Folder("y1yb5X3W") # Older Folder
new_pd = dataiku.Folder("QbhHPgq3") # New Folder

MAX_RETRIES = 3
N_THREADS =  os.cpu_count() - 1

start = time.time()
print(f"[INFO] Threads to be used: {N_THREADS}")

# --------------------------------------------------
# Collect partitions & total file count
# --------------------------------------------------
FILTER_KEYS = []
#FILTER_KEYS = ["|diskspace|", "|filesystem|"]
all_paths = collect_all_paths(old_pd, n_threads=N_THREADS, FILTER_KEYS=FILTER_KEYS)

print(f"[INFO] Total files to migrate: {len(all_paths)}")

# --------------------------------------------------
# Migrate the data
# --------------------------------------------------
results = Parallel(
    n_jobs=N_THREADS,
    backend="threading",
    prefer="threads",
)(
    delayed(migrate_one_file)(old_pd, new_pd, True, p)
    for p in all_paths
)

elapsed = time.time() - start

skipped = sum(1 for r in results if r.get("skipped"))
success = sum(1 for r in results if r.get("ok") and not r.get("skipped"))
failed = sum(1 for r in results if not r.get("ok"))

print("[DONE]")
print(f"[SUMMARY] Migrated....: {success}/{len(all_paths)}")
print(f"[SUMMARY] Skipped.....: {skipped}/{len(all_paths)}")
print(f"[SUMMARY] Failed......: {failed}/{len(all_paths)}")
print(f"[SUMMARY] Time: {elapsed/60:.1f} min")
print(f"[SUMMARY] Rate: {success/elapsed:.1f} files/sec")

[INFO] Threads to be used: 7
